In [1]:
import os, re, json, csv, random, numpy as np
import platform, time
import torch
from collections import Counter
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
import subprocess, transformers, bitsandbytes, accelerate
import medmnist
from medmnist import INFO
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_recall_fscore_support, confusion_matrix

from dotenv import load_dotenv
from huggingface_hub import login
load_dotenv()
login(token=os.getenv("HF_TOKEN"))

MODEL_ID = "google/medgemma-1.5-4b-it"
FLAG = "pneumoniamnist"
RUN = f"{FLAG}_medgemma15-4b_4bit" 
SIZE = 224
SEED = 42
MAX_NEW_TOKENS = 24
LIMIT = None 

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

INFO_D = INFO[FLAG]
CLASS_NAMES = [INFO_D["label"][str(i)] for i in range(len(INFO_D["label"]))]
N_CLASSES = len(CLASS_NAMES)

W0726 00:37:51.891000 5896 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
def compute_metrics(y_true, y_pred, class_names=None, n_classes=None):
    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()
    if n_classes is None:
        n_classes = int(max(y_true.max(), y_pred.max())) + 1
    labels = list(range(n_classes))

    p_mac, r_mac, f_mac, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", labels=labels, zero_division=0)
    p_w, r_w, f_w, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", labels=labels, zero_division=0)
    p_c, r_c, f_c, support = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, zero_division=0)

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_macro": p_mac, "recall_macro": r_mac, "f1_macro": f_mac,
        "precision_weighted": p_w, "recall_weighted": r_w, "f1_weighted": f_w,
        "per_class": {"precision": p_c.tolist(), "recall": r_c.tolist(),
                      "f1": f_c.tolist(), "support": support.tolist()},
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels).tolist(),
        "class_names": list(class_names) if class_names else None,
    }

def print_metrics(results, title=""):
    if title: print(f"\n--- {title} ---")
    for k in ["accuracy", "balanced_accuracy", "precision_macro",
              "recall_macro", "f1_macro", "f1_weighted"]:
        print(f"{k:<20}: {results[k]:.4f}")
    names, pc = results.get("class_names"), results["per_class"]
    print(f"\n{'class':<45}{'prec':>7}{'rec':>8}{'f1':>8}{'n':>7}")
    for i in range(len(pc["f1"])):
        label = names[i] if names else str(i)
        print(f"{label[:44]:<45}{pc['precision'][i]:>7.3f}"
              f"{pc['recall'][i]:>8.3f}{pc['f1'][i]:>8.3f}{pc['support'][i]:>7}")
    print("\nConfusion matrix (rows = true, cols = pred):")
    for row in results["confusion_matrix"]:
        print("  " + " ".join(f"{v:>5}" for v in row))

def majority_baseline(y_true, class_names=None, n_classes=None):
    y_true = np.asarray(y_true).flatten()
    y_pred = np.full_like(y_true, np.bincount(y_true).argmax())
    return compute_metrics(y_true, y_pred, class_names, n_classes)

In [3]:
DataClass = getattr(medmnist, INFO_D["python_class"])

test_ds = DataClass(split="test", download=True, size=SIZE, as_rgb=True)
train_ds = DataClass(split="train", download=True, size=SIZE, as_rgb=True)

test_labels = np.asarray(test_ds.labels).flatten()
train_labels = np.asarray(train_ds.labels).flatten()

print(f"test: {len(test_ds)}  train: {len(train_ds)}")
print("test class counts:", Counter(test_labels.tolist()))
print_metrics(majority_baseline(test_labels, CLASS_NAMES, N_CLASSES),
              "majority-class baseline (test)")

100%|██████████| 214M/214M [03:15<00:00, 1.09MB/s]


test: 624  train: 4708
test class counts: Counter({1: 390, 0: 234})

--- majority-class baseline (test) ---
accuracy            : 0.6250
balanced_accuracy   : 0.5000
precision_macro     : 0.3125
recall_macro        : 0.5000
f1_macro            : 0.3846
f1_weighted         : 0.4808

class                                           prec     rec      f1      n
normal                                         0.000   0.000   0.000    234
pneumonia                                      0.625   1.000   0.769    390

Confusion matrix (rows = true, cols = pred):
      0   234
      0   390


In [4]:
"""
used 4-bit NF4 quantization (bitsandbytes) that uses ~3GB VRAM for weights, 
and leaves room for the 256 image tokens + KV cache
"""

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, 
    quantization_config=bnb, 
    device_map="auto", 
    dtype=torch.bfloat16
)
model.eval()
processor = AutoProcessor.from_pretrained(MODEL_ID)

print(next(model.parameters()).device, next(model.parameters()).dtype)
print(f"VRAM allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

C:\Users\Kawshik\anaconda3\envs\dl_cv\lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


cuda:0 torch.bfloat16
VRAM allocated: 3.23 GB


In [5]:
LETTERS = ["A", "B", "C", "D", "E", "F", "G"]
OPTIONS = "\n".join(f"{L}. {n}" for L, n in zip(LETTERS, CLASS_NAMES))

INSTRUCTION = (
    "You are shown a chest X-ray image.\n"
    "Classify it into exactly one of the following categories:\n\n"
    f"{OPTIONS}\n\n"
    "Respond with only the single letter of the correct category (A-B). Do not provide any explanation or extra text."
)

SYNONYMS = {
    0: [r"\bnormal\b", r"no pneumonia", r"no evidence", r"healthy", r"unremarkable", r"clear"],
    1: [r"pneumonia", r"pneumonic", r"consolidation", r"infiltrat", r"opacit", r"abnormal"],
}

def parse_answer(text):
    """Return class index, or -1 if unparseable."""
    t = text.strip()
    m = re.match(r"^[\s\*\"'`]*\(?([A-G])[\)\.\:,\*]?(\s|$)", t, re.I)
    if m:
        return LETTERS.index(m.group(1).upper())
    m = re.search(r"\*\*([A-G])[\.\)\:]", t)
    if m:
        return LETTERS.index(m.group(1).upper())
    low = t.lower()
    for i, name in enumerate(CLASS_NAMES):
        if name.lower() in low:
            return i
    hits = {i for i, syns in SYNONYMS.items() if any(re.search(s, low) for s in syns)}
    if len(hits) == 1:
        return hits.pop()
    return -1

In [6]:
# for few-shot
rng = np.random.RandomState(SEED)
shot_idx = [int(rng.choice(np.flatnonzero(train_labels == c))) for c in range(N_CLASSES)]
print("exemplar indices:", shot_idx, "labels:", train_labels[shot_idx].tolist())

def build_messages(image, few_shot=False):
    msgs = []
    if few_shot:
        for si in shot_idx:
            img, lab = train_ds[si]
            c = int(np.asarray(lab).flatten()[0])
            msgs.append({"role": "user", "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": INSTRUCTION}]})
            msgs.append({"role": "assistant", "content": [
                {"type": "text", "text": LETTERS[c]}]})
    msgs.append({"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": INSTRUCTION}]})
    msgs.append({"role": "assistant", "content": [
        {"type": "text", "text": "Answer:"}]})
    return msgs

exemplar indices: [4303, 1165] labels: [0, 1]


In [7]:
# inference 
@torch.inference_mode()
def predict_one(image, few_shot=False):
    msgs = build_messages(image, few_shot)
    
    # print(repr(processor.apply_chat_template(
    #     msgs, add_generation_prompt=False, continue_final_message=True, tokenize=False)))
    
    inputs = processor.apply_chat_template(
        msgs, add_generation_prompt=False, continue_final_message=True,
        tokenize=True, return_dict=True, return_tensors="pt").to(model.device)
    if "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)
    n_in = inputs["input_ids"].shape[-1]
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    return processor.decode(out[0][n_in:], skip_special_tokens=True)

In [8]:
def score(preds, y_true, tag):
    preds = np.asarray(preds); y_true = np.asarray(y_true)
    ok = preds != -1
    n_fail = int((~ok).sum())
    print(f"\n===== {tag} =====")
    print(f"parse failures: {n_fail}/{len(preds)} ({100*n_fail/len(preds):.2f}%)")
    res = compute_metrics(y_true[ok], preds[ok], CLASS_NAMES, N_CLASSES)
    print_metrics(res, tag)
    res["n_parse_fail"] = n_fail
    res["n_total"] = int(len(preds))
    print("pred distribution:", Counter(preds[ok].tolist()))
    return res

In [9]:
def run_eval(few_shot=False, limit=None, stop_after_fails=None, dump_csv=None):
    n = len(test_ds) if limit is None else min(limit, len(test_ds))
    raw, preds, lat = [], [], []
    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    stopped = False
    for i in range(n):
        img, _ = test_ds[i]
        ts = time.perf_counter()
        txt = predict_one(img, few_shot)
        torch.cuda.synchronize()
        lat.append(time.perf_counter() - ts)
        raw.append(txt)
        preds.append(parse_answer(txt))
        if (i + 1) % 250 == 0:
            print(f"  {i+1}/{n}  fails={preds.count(-1)}  "
                  f"{np.mean(lat[-250:]):.2f} s/img", flush=True)
        if stop_after_fails is not None and preds.count(-1) >= stop_after_fails:
            print(f"stopping at i={i}: {preds.count(-1)} parse failures", flush=True)
            stopped = True
            break

    if dump_csv:
        with open(dump_csv, "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow(["idx", "true", "pred", "parsed_ok", "raw"])
            for j, (t, p) in enumerate(zip(raw, preds)):
                w.writerow([j, int(test_labels[j]), p, int(p != -1), t])
        print(f"wrote {dump_csv} ({len(raw)} rows)", flush=True)

    prof = {
        "wall_time_s": time.perf_counter() - t0,
        "latency_mean_s": float(np.mean(lat)),
        "latency_median_s": float(np.median(lat)),
        "latency_p95_s": float(np.percentile(lat, 95)),
        "peak_vram_gb": torch.cuda.max_memory_allocated() / 1e9,
        "n_images": len(raw),
        "stopped_early": stopped,
    }
    return np.array(preds), raw, test_labels[:len(raw)], prof

In [10]:
results = {}
preds, raw, y, prof = run_eval(few_shot=False, limit=None, dump_csv=f"raw_zero-shot_{RUN}.csv")
print(f"elapsed {prof['wall_time_s']/60:.1f} min | "
      f"{prof['latency_mean_s']:.2f} s/img | peak {prof['peak_vram_gb']:.2f} GB")
r = score(preds, y, "zero-shot")
r["profile"] = prof; r["raw"] = raw; r["preds"] = preds.tolist()
results["zero-shot"] = r

import gc; gc.collect(); torch.cuda.empty_cache()

[transformers] Deprecated: `processor.image_token` will switch from returning `tokenizer.image_token` to `tokenizer.boi_token` in v5.11.
C:\Users\Kawshik\anaconda3\envs\dl_cv\lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  250/624  fails=0  1.70 s/img
  500/624  fails=0  1.69 s/img
wrote raw_zero-shot_pneumoniamnist_medgemma15-4b_4bit.csv (624 rows)
elapsed 17.6 min | 1.70 s/img | peak 3.36 GB

===== zero-shot =====
parse failures: 0/624 (0.00%)

--- zero-shot ---
accuracy            : 0.8478
balanced_accuracy   : 0.8671
precision_macro     : 0.8444
recall_macro        : 0.8671
f1_macro            : 0.8447
f1_weighted         : 0.8502

class                                           prec     rec      f1      n
normal                                         0.729   0.944   0.823    234
pneumonia                                      0.960   0.790   0.866    390

Confusion matrix (rows = true, cols = pred):
    221    13
     82   308
pred distribution: Counter({1: 321, 0: 303})


In [11]:
preds, raw, y, prof = run_eval(few_shot=True, limit=None, dump_csv=f"raw_2-shot_{RUN}.csv")
print(f"elapsed {prof['wall_time_s']/60:.1f} min | "
      f"{prof['latency_mean_s']:.2f} s/img | peak {prof['peak_vram_gb']:.2f} GB")
r = score(preds, y, "few-shot")
r["profile"] = prof; r["raw"] = raw; r["preds"] = preds.tolist()
results["few-shot"] = r

import gc; gc.collect(); torch.cuda.empty_cache()

C:\Users\Kawshik\anaconda3\envs\dl_cv\lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  250/624  fails=0  10.49 s/img
  500/624  fails=0  10.05 s/img
wrote raw_2-shot_pneumoniamnist_medgemma15-4b_4bit.csv (624 rows)
elapsed 106.5 min | 10.24 s/img | peak 3.59 GB

===== few-shot =====
parse failures: 0/624 (0.00%)

--- few-shot ---
accuracy            : 0.6154
balanced_accuracy   : 0.6923
precision_macro     : 0.7468
recall_macro        : 0.6923
f1_macro            : 0.6083
f1_weighted         : 0.5951

class                                           prec     rec      f1      n
normal                                         0.494   1.000   0.661    234
pneumonia                                      1.000   0.385   0.556    390

Confusion matrix (rows = true, cols = pred):
    234     0
    240   150
pred distribution: Counter({0: 474, 1: 150})


In [12]:
try:
    gpu = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
         "--format=csv,noheader"], text=True).strip()
except Exception:
    gpu = torch.cuda.get_device_name(0)

env = {
    "gpu": gpu,
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "transformers": transformers.__version__,
    "bitsandbytes": bitsandbytes.__version__,
    "accelerate": accelerate.__version__,
    "python": platform.python_version(),
    "platform": platform.platform(),
}

out = f"results_{RUN}.json"
payload = {
    "model": MODEL_ID,
    "dataset": FLAG,
    "quantization": "nf4-4bit-double",
    "compute_dtype": "bfloat16",
    "decoding": "greedy",
    "max_new_tokens": MAX_NEW_TOKENS,
    "batch_size": 1,
    "seed": SEED,
    "image_size_in": SIZE,
    "n_test": int(len(test_ds)),
    "limit": LIMIT,
    "shot_indices": shot_idx,
    "prompt": INSTRUCTION,
    "env": env,
    "baseline": {k: v for k, v in
                 majority_baseline(test_labels, CLASS_NAMES, N_CLASSES).items()
                 if k != "class_names"},
    "conditions": {k: {kk: vv for kk, vv in v.items() if kk != "raw"}
                   for k, v in results.items()},
    "raw_outputs": {k: v["raw"] for k, v in results.items()},
}
with open(out, "w") as f:
    json.dump(payload, f, indent=2)
print("saved", out)

saved results_pneumoniamnist_medgemma15-4b_4bit.json


In [13]:
print(f"{'condition':<12}{'f1_macro':>10}{'bal_acc':>10}{'acc':>8}"
      f"{'s/img':>8}{'VRAM':>8}{'fail%':>8}")
b = majority_baseline(test_labels, CLASS_NAMES, N_CLASSES)
print(f"{'majority':<12}{b['f1_macro']:>10.4f}{b['balanced_accuracy']:>10.4f}"
      f"{b['accuracy']:>8.4f}{'-':>8}{'-':>8}{'-':>8}")
for tag, r in results.items():
    p = r["profile"]
    print(f"{tag:<12}{r['f1_macro']:>10.4f}{r['balanced_accuracy']:>10.4f}"
          f"{r['accuracy']:>8.4f}{p['latency_mean_s']:>8.2f}"
          f"{p['peak_vram_gb']:>8.2f}"
          f"{100*r['n_parse_fail']/r['n_total']:>8.2f}")

condition     f1_macro   bal_acc     acc   s/img    VRAM   fail%
majority        0.3846    0.5000  0.6250       -       -       -
zero-shot       0.8447    0.8671  0.8478    1.70    3.36    0.00
few-shot        0.6083    0.6923  0.6154   10.24    3.59    0.00
